In [ ]:
!git clone https://github.com/Felix-Schwer/rainfall_kf.git
%cd rainfall_kf
!pip install -e .

[Errno 2] No such file or directory: 'rainfall_kf'
/content
Obtaining file:///content
ERROR: file:///content does not appear to be a Python project: neither 'setup.py' nor 'pyproject.toml' found.


In [6]:
import numpy as np
from rainfall_kf.models.lorenz import LorenzParameters, LorenzTransition, LorenzObservation
from rainfall_kf.enkf import EnsembleKalmanFilter

DT = 0.01
times = np.arange(0, 2, DT)
sim_state0 = np.array([[1.0], [1.0], [1.0]])
lorenz_params = LorenzParameters()

sim_obs = np.empty((2, len(times)))
state  = sim_state0
for i, t in enumerate(times):
    state = LorenzTransition(state, lorenz_params, dt=DT) + 0.1 * np.random.randn(3, 1)  # Add process noise to the true state
    sim_obs[:, i] = (LorenzObservation(state) + 0.1 * np.random.randn(2, 1)).flatten()

AttributeError: 'EnsembleKalmanFilter' object has no attribute 'rng'

In [ ]:
enkf = EnsembleKalmanFilter(TransitionEquation=lambda states: LorenzTransition(states, lorenz_params, dt=DT),
                           ObservationEquation=LorenzObservation,
                           Q=np.diag([0.1, 0.1, 0.1]),
                           R=np.diag([0.1, 0.1]))

initial_ensemble = sim_state0 + 0.1 * np.random.randn(3, 100)  # Perturb initial state to create ensemble

result = enkf.run(initial_ensemble=initial_ensemble, times=times, observations=sim_obs)

In [ ]:
fig = result.plot_ensembles()
axes  = fig.get_axes()
axes[0].set_title("Ensemble trajectories for Lorenz attractor")
axes[2].set_xlabel("Time")
axes[0].set_ylabel("x")
axes[1].set_ylabel("y")
axes[2].set_ylabel("z")

fig = result.plot_observations()
ax = fig.get_axes()[0]
ax.set_title("Predicted observations vs measurements")
ax.set_xlabel("Time")
ax.set_ylabel("x")

fig = result.plot_innovations()
ax = fig.get_axes()[0]
ax.set_ylabel("x")
ax.set_xlabel("Time")

fig = result.plot_whitened_innovations()
ax = fig.get_axes()[0]
ax.set_xlabel("Time")